# IBS and Yang–Zhang Volatility Walk-Forward

Compares Internal Bar Strength mean reversion, a Yang–Zhang volatility rule, and a combined strategy. Runs chronological training, validation, and test windows on daily Yahoo Finance prices for SPY, with URTH as the benchmark. Both symbols and the study dates are configurable.


In [ ]:
import pandas as pd
import numpy as np
from _market_data import download_ohlcv
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
SYMBOL = 'SPY'
BENCHMARK = 'URTH'
START_DATE = '2015-01-01'
END_DATE = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
INITIAL_TRAIN_DAYS = 252 * 2
VALIDATION_DAYS = 126
TEST_DAYS = 63
REOPTIMIZATION_DAYS = 63

def download_ohlc_data(ticker, start_date, end_date):
    try:
        prices = download_ohlcv(ticker, start_date, end_date)
    except RuntimeError as error:
        print(error)
        return None
    frame = prices.rename(columns={'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close', 'Adj Close': 'adj_close', 'Volume': 'volume'})
    if 'adj_close' not in frame.columns:
        raise RuntimeError(f'Yahoo Finance returned no adjusted close for {ticker}.')
    frame = frame.dropna(subset=['adj_close'])
    return frame.rename_axis('date').reset_index() if len(frame) >= 500 else None

def calculate_ibs(df):
    intraday_range = df['high'] - df['low']
    ibs = (df['close'] - df['low']) / intraday_range
    ibs = ibs.clip(0, 1)
    return ibs

def calculate_yang_zhang_volatility(df, window=20):
    log_open_prev_close = np.log(df['open'] / df['close'].shift(1))
    log_close_open = np.log(df['close'] / df['open'])
    log_close_close = np.log(df['close'] / df['close'].shift(1))
    log_high_close = np.log(df['high'] / df['close'])
    log_low_close = np.log(df['low'] / df['close'])
    overnight_var = log_open_prev_close.rolling(window).var()
    rs_var = (log_high_close * (log_high_close - log_close_open) + log_low_close * (log_low_close - log_close_open)).rolling(window).mean()
    close_var = log_close_close.rolling(window).var()
    k = 0.34 / (1.34 + (window + 1) / (window - 1))
    yz_var = overnight_var + k * close_var + (1 - k) * rs_var
    yz_vol = np.sqrt(yz_var * 252)
    return yz_vol

def create_features_target(df):
    """Build lagged signals and forward returns without filling price gaps."""
    result = pd.DataFrame(index=df.index)
    result['date'] = df['date']
    result['forward_return'] = np.log(df['adj_close'] / df['adj_close'].shift(1)).shift(-1)
    ibs = calculate_ibs(df)
    result['ibs_lag1'] = ibs.shift(1)
    result['ibs_lag2'] = ibs.shift(2)
    result['ibs_ma5'] = ibs.shift(1).rolling(5).mean()
    yz_vol = calculate_yang_zhang_volatility(df, window=20)
    result['yz_vol_lag1'] = yz_vol.shift(1)
    result['vol_percentile'] = yz_vol.shift(1).rolling(20).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1] if len(x) > 0 else np.nan)
    result['ibs_vol_interaction'] = result['ibs_lag1'] * result['yz_vol_lag1']
    result['normalized_range_lag1'] = ((df['high'] - df['low']) / df['close']).shift(1)
    result['momentum_5d'] = np.log(df['adj_close'].shift(1) / df['adj_close'].shift(6))
    result = result.dropna()
    print(f'Features generated: {len(result)} complete rows')
    print(f'Columns: {list(result.columns)}')
    return result

def strategy_ibs_mean_reversion(features, ibs_low=0.2, ibs_high=0.8):
    signals = pd.Series(0, index=features.index)
    signals[features['ibs_lag1'] < ibs_low] = 1
    signals[features['ibs_lag1'] > ibs_high] = -1
    return signals

def strategy_vol_regime(features, vol_low_pct=0.3, vol_high_pct=0.7):
    signals = pd.Series(0, index=features.index)
    low_vol = features['vol_percentile'] < vol_low_pct
    signals[low_vol] = 1
    high_vol = features['vol_percentile'] > vol_high_pct
    signals[high_vol] = -1
    return signals

def strategy_combined(features, ibs_low=0.25, ibs_high=0.75, vol_low_pct=0.4, vol_high_pct=0.6):
    signals = pd.Series(0.0, index=features.index)
    ibs_low_cond = features['ibs_lag1'] < ibs_low
    ibs_high_cond = features['ibs_lag1'] > ibs_high
    vol_low_cond = features['vol_percentile'] < vol_low_pct
    vol_high_cond = features['vol_percentile'] > vol_high_pct
    signals[ibs_low_cond & vol_low_cond] = 1.0
    signals[ibs_low_cond & vol_high_cond] = 0.5
    signals[ibs_high_cond & vol_low_cond] = -0.5
    signals[ibs_high_cond & vol_high_cond] = -1.0
    return signals

def walk_forward_analysis(features, strategy_func, param_grid, benchmark_returns=None):
    results = {'blocks': [], 'equity_curve': [], 'params_history': [], 'metrics_history': []}
    n = len(features)
    start_idx = 0
    print('\n' + '=' * 80)
    print('INICIANDO WALK-FORWARD ANALYSIS')
    print('=' * 80)
    block_num = 0
    while start_idx + INITIAL_TRAIN_DAYS + VALIDATION_DAYS + TEST_DAYS < n:
        block_num += 1
        train_end = start_idx + INITIAL_TRAIN_DAYS
        val_end = train_end + VALIDATION_DAYS
        test_end = min(val_end + TEST_DAYS, n)
        train_data = features.iloc[start_idx:train_end]
        val_data = features.iloc[train_end:val_end]
        test_data = features.iloc[val_end:test_end]
        print(f"\n{'=' * 60}")
        print(f'BLOCK {block_num}')
        print(f"{'=' * 60}")
        print(f"Train: {train_data['date'].iloc[0]} to {train_data['date'].iloc[-1]} ({len(train_data)} days)")
        print(f"Val:   {val_data['date'].iloc[0]} to {val_data['date'].iloc[-1]} ({len(val_data)} days)")
        print(f"Test:  {test_data['date'].iloc[0]} to {test_data['date'].iloc[-1]} ({len(test_data)} days)")
        print('\nPhase 1: tuning parameters on the training window...')
        best_sharpe = -np.inf
        best_params = None
        from itertools import product
        param_combinations = [dict(zip(param_grid.keys(), v)) for v in product(*param_grid.values())]
        for params in param_combinations:
            signals = strategy_func(train_data, **params)
            strategy_returns = signals * train_data['forward_return']
            sharpe = calculate_sharpe_ratio(strategy_returns)
            if sharpe > best_sharpe:
                best_sharpe = sharpe
                best_params = params
        print(f'Selected parameters: {best_params}')
        print(f'Sharpe in-sample (train): {best_sharpe:.3f}')
        print('\nPhase 2: evaluating validation returns...')
        val_signals = strategy_func(val_data, **best_params)
        val_returns = val_signals * val_data['forward_return']
        val_sharpe = calculate_sharpe_ratio(val_returns)
        print(f'Validation Sharpe ratio: {val_sharpe:.3f}')
        print(f'Change from train to validation: {best_sharpe - val_sharpe:.3f}')
        print('\nPhase 3: out-of-sample test...')
        test_signals = strategy_func(test_data, **best_params)
        test_returns = test_signals * test_data['forward_return']
        test_sharpe = calculate_sharpe_ratio(test_returns)
        print(f'Test Sharpe ratio: {test_sharpe:.3f}')
        print(f'Change from validation to test: {val_sharpe - test_sharpe:.3f}')
        if benchmark_returns is not None:
            test_dates = test_data['date']
            bench_test = benchmark_returns[benchmark_returns.index.isin(test_dates)]
            bench_sharpe = calculate_sharpe_ratio(bench_test)
            print(f'Sharpe benchmark (test): {bench_sharpe:.3f}')
            print(f'Outperformance: {test_sharpe - bench_sharpe:.3f}')
        block_results = {'block_num': block_num, 'train_start': train_data['date'].iloc[0], 'test_end': test_data['date'].iloc[-1], 'best_params': best_params, 'sharpe_train': best_sharpe, 'sharpe_val': val_sharpe, 'sharpe_test': test_sharpe, 'degradation_train_val': best_sharpe - val_sharpe, 'degradation_val_test': val_sharpe - test_sharpe, 'test_returns': test_returns, 'test_dates': test_data['date']}
        results['blocks'].append(block_results)
        results['params_history'].append(best_params)
        start_idx = start_idx
        start_idx = val_end
    print('\n' + '=' * 80)
    print('WALK-FORWARD ANALYSIS COMPLETE')
    print('=' * 80)
    return results

def calculate_sharpe_ratio(returns, periods_per_year=252):
    if len(returns) == 0 or returns.std() == 0:
        return 0.0
    mean_return = returns.mean() * periods_per_year
    std_return = returns.std() * np.sqrt(periods_per_year)
    return mean_return / std_return if std_return > 0 else 0.0

def calculate_metrics(returns, benchmark_returns=None):
    metrics = {}
    metrics['total_return'] = np.expm1(returns.sum())
    n_years = len(returns) / 252
    metrics['annual_return'] = (1 + metrics['total_return']) ** (1 / n_years) - 1 if n_years > 0 else 0
    metrics['annual_volatility'] = returns.std() * np.sqrt(252)
    metrics['sharpe_ratio'] = calculate_sharpe_ratio(returns)
    cumulative = np.exp(returns.cumsum())
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    metrics['max_drawdown'] = drawdown.min()
    metrics['win_rate'] = (returns > 0).sum() / len(returns) if len(returns) > 0 else 0
    if benchmark_returns is not None:
        aligned_benchmark = benchmark_returns.reindex(returns.index).dropna()
        aligned_strategy = returns.reindex(aligned_benchmark.index)
        bench_total = np.expm1(aligned_benchmark.sum())
        metrics['excess_total_return'] = np.expm1(aligned_strategy.sum()) - bench_total if len(aligned_strategy) else np.nan
        if len(aligned_strategy) > 1:
            covariance = aligned_strategy.cov(aligned_benchmark)
            benchmark_var = aligned_benchmark.var()
            metrics['beta'] = covariance / benchmark_var if benchmark_var > 0 else 0
        else:
            metrics['beta'] = np.nan
    return metrics

def main():
    """Compare the three walk-forward strategy variants."""
    print('=' * 80)
    print('IBS AND YANG–ZHANG VOLATILITY')
    print('Walk-forward return analysis')
    print('=' * 80)
    print('\n[1/5] Downloading prices...')
    selected_ticker = SYMBOL
    print(f'\nSelected symbol: {selected_ticker}')
    df = download_ohlc_data(selected_ticker, START_DATE, END_DATE)
    if df is None or len(df) < 1000:
        print(f'ERROR: Insufficient observations for {selected_ticker}')
        return
    print(f"Downloaded: {len(df)} days from {df['date'].iloc[0]} through {df['date'].iloc[-1]}")
    print(f'\nDownloading benchmark {BENCHMARK}...')
    benchmark_df = download_ohlc_data(BENCHMARK, START_DATE, END_DATE)
    if benchmark_df is None:
        print(f'Missing benchmark history for {BENCHMARK}')
        benchmark_returns = None
    else:
        benchmark_returns = np.log(benchmark_df['adj_close'].shift(-1) / benchmark_df['adj_close']).dropna()
        benchmark_returns.index = pd.DatetimeIndex(benchmark_df.loc[benchmark_returns.index, 'date'])
        print(f'Benchmark downloaded: {len(benchmark_df)} days')
    print('\n[2/5] Computing features (IBS, Yang-Zhang Vol)...')
    features = create_features_target(df)
    if len(features) < 500:
        print('ERROR: Insufficient complete features')
        return
    print(f'\nFeature summary:')
    print(features[['ibs_lag1', 'yz_vol_lag1', 'forward_return']].describe())
    print('\n[3/5] Defining parameter grids...')
    ibs_param_grid = {'ibs_low': [0.15, 0.2, 0.25, 0.3], 'ibs_high': [0.7, 0.75, 0.8, 0.85]}
    vol_param_grid = {'vol_low_pct': [0.2, 0.3, 0.4], 'vol_high_pct': [0.6, 0.7, 0.8]}
    combined_param_grid = {'ibs_low': [0.2, 0.25, 0.3], 'ibs_high': [0.7, 0.75, 0.8], 'vol_low_pct': [0.3, 0.4], 'vol_high_pct': [0.6, 0.7]}
    print('\n[4/5] Running Walk-Forward Analysis...')
    print('\n' + '=' * 80)
    print('STRATEGY 1: IBS MEAN REVERSION')
    print('=' * 80)
    results_ibs = walk_forward_analysis(features, strategy_ibs_mean_reversion, ibs_param_grid, benchmark_returns)
    print('\n' + '=' * 80)
    print('STRATEGY 2: VOLATILITY REGIME')
    print('=' * 80)
    results_vol = walk_forward_analysis(features, strategy_vol_regime, vol_param_grid, benchmark_returns)
    print('\n' + '=' * 80)
    print('STRATEGY 3: IBS + VOLATILITY COMBINED')
    print('=' * 80)
    results_combined = walk_forward_analysis(features, strategy_combined, combined_param_grid, benchmark_returns)
    print('\n[5/5] Analyzing results...')

    def concatenate_test_returns(results):
        all_returns = pd.Series(dtype=float)
        for block in results['blocks']:
            test_returns = pd.Series(block['test_returns'].values, index=block['test_dates'].values)
            all_returns = pd.concat([all_returns, test_returns])
        return all_returns
    returns_ibs = concatenate_test_returns(results_ibs)
    returns_vol = concatenate_test_returns(results_vol)
    returns_combined = concatenate_test_returns(results_combined)
    print('\n' + '=' * 80)
    print('FINAL RESULTS (OUT-OF-SAMPLE AGGREGATED)')
    print('=' * 80)
    print('\nStrategy 1: IBS Mean Reversion')
    metrics_ibs = calculate_metrics(returns_ibs, benchmark_returns)
    for key, value in metrics_ibs.items():
        print(f'  {key}: {value:.4f}')
    print('\nStrategy 2: Volatility Regime')
    metrics_vol = calculate_metrics(returns_vol, benchmark_returns)
    for key, value in metrics_vol.items():
        print(f'  {key}: {value:.4f}')
    print('\nStrategy 3: Combined IBS + Vol')
    metrics_combined = calculate_metrics(returns_combined, benchmark_returns)
    for key, value in metrics_combined.items():
        print(f'  {key}: {value:.4f}')
    if benchmark_returns is not None:
        print(f'\nBenchmark ({BENCHMARK})')
        test_dates = returns_combined.index
        bench_aligned = benchmark_returns[benchmark_returns.index.isin(test_dates)]
        metrics_bench = calculate_metrics(bench_aligned)
        for key, value in metrics_bench.items():
            if key not in ['excess_total_return', 'beta']:
                print(f'  {key}: {value:.4f}')
    print('=' * 80)
    print('ANALYSIS COMPLETE')
    print('=' * 80)
if __name__ == '__main__':
    main()
